In [1]:
# predict.py
import joblib
import pandas as pd
import numpy as np

In [2]:
# 1. Load your AI Brain and your Math Translator (Do this once when the server starts)
try:
    model = joblib.load('../training/hybrid_water_model.pkl')
    scaler = joblib.load('../data/scaler.pkl')
    print("✅ AI Model and Scaler loaded successfully.")
except FileNotFoundError:
    print("❌ Error: Could not find the .pkl files. Make sure they are in the same folder.")

✅ AI Model and Scaler loaded successfully.


In [5]:
def get_water_demand(hour, day, month, temp_c, rain_mm, lag_1, lag_24):
    """
    Takes real-time data from the Digital Twin simulation or sensors 
    and returns the predicted water demand in Liters per Second.
    """
    
    # 1. Package the incoming data exactly how the scaler expects it (8 columns)
    # We put a temporary '0.0' for Current_Demand because we don't know it yet!
    input_data = pd.DataFrame([[hour, day, month, temp_c, rain_mm, lag_1, lag_24, 0.0]], 
                              columns=['Hour_of_Day', 'Day_of_Week', 'Month', 
                                       'Temperature_C', 'Rainfall_mm', 
                                       'Lag_1_Demand', 'Lag_24_Demand', 'Current_Demand'])
    
    # 2. Translate (Scale) the human data into AI data (0 to 1)
    scaled_array = scaler.transform(input_data)
    
    # 3. Strip away the fake target column so we only feed the 7 features to the model
    scaled_features = scaled_array[:, :-1] 
    
    # 4. Get the AI's prediction (It will be a number between 0 and 1)
    scaled_prediction = model.predict(scaled_features)[0]
    
    # 5. Translate the AI's answer back into real Liters/second
    # We create a dummy array, put the prediction in the last column, and reverse the scale
    dummy_out = np.zeros((1, 8))
    dummy_out[0, -1] = scaled_prediction
    real_prediction = scaler.inverse_transform(dummy_out)[0, -1]
    
    # Return a clean, rounded number for the dashboard
    return round(real_prediction, 2)

In [9]:
def check_for_leaks(predicted_demand, actual_sensor_demand, alert_threshold=20.0):
    """
    Compares AI prediction to live sensor data to detect anomalies (leaks).
    Returns a dictionary with the alert status and details for the dashboard.
    """
    # Calculate the unexplained extra water
    difference = actual_sensor_demand - predicted_demand
    
    # We only care if actual demand is drastically HIGHER than predicted
    # (If it's lower, that might be a sensor failure, not a leak)
    is_leaking = difference > alert_threshold
    
    return {
        "alert_triggered": is_leaking,
        "predicted_flow": round(predicted_demand, 2),
        "actual_flow": round(actual_sensor_demand, 2),
        "unexplained_water_loss": round(difference, 2) if is_leaking else 0.0,
        "status_message": "🚨 LEAK DETECTED!" if is_leaking else "✅ System Normal"
    }

In [10]:
# ==========================================
# TEST BLOCK: You can run this file directly to test it
# ==========================================
if __name__ == "__main__":
    print("\n--- Running Live Prediction Test ---")
    # Simulating a hot Tuesday afternoon (Day 1) in June (Month 6) at 2 PM (Hour 14)
    # Yesterday's flow was 120 L/s, and the last hour's flow was 125 L/s.
    predicted_flow = get_water_demand(
        hour=14, 
        day=1, 
        month=6, 
        temp_c=35.5, 
        rain_mm=0.0, 
        lag_1=125.0, 
        lag_24=120.0
    )
    
    print(f"🔮 Predicted Water Demand: {predicted_flow} Liters/second")


--- Running Live Prediction Test ---
🔮 Predicted Water Demand: 116.91 Liters/second


c:\Users\Acer\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MLPRegressor was fitted with feature names
  warnings.warn(
c:\Users\Acer\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
